In [2]:
# run cell to install prokbert lib (if necessary)
# pip install git+https://github.com/nbrg-ppcu/prokbert.git --quiet

In [1]:
import torch
import datasets
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, balanced_accuracy_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from prokbert.tokenizer import LCATokenizer
from prokbert.models import ProkBertForSequenceClassification

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

from prokbert.sequtils import load_contigs, segment_sequences
from prokbert.tokenizer import LCATokenizer

In [2]:
num_cores = 8
batch_size = 16

base_dir = '/home/datasets/experiments/experiment-bioinfo/'

In [3]:
train_ds = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_TRAINING")
test_ds  = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_VALIDATION")

2026-10-08 11:43:35,234 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/neuralbioinfo/PhaStyle-SequenceDB/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-08 11:43:35,237 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-08 11:43:35,261 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/neuralbioinfo/PhaStyle-SequenceDB/a7772eef14b1b056ee9b14448a71b8d182cfc4e2/README.md "HTTP/1.1 200 OK"
2026-10-08 11:43:35,394 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/neuralbioinfo/PhaStyle-SequenceDB/resolve/a7772eef14b1b056ee9b14448a71b8d182cfc4e2/PhaStyle-SequenceDB.py "HTTP/1.1 404 Not Found"
2026-10-08 11:43:35,760 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/neuralbioinfo/PhaStyle-SequenceDB/neuralbioinfo/PhaStyle-SequenceDB.py "HTTP/1.1 404 Not Found"
2026-10-08 11:43

In [4]:
print(f"train examples: {len(train_ds)}, columns: {train_ds.column_names}")
print(f"test examples: {len(test_ds)}, columns: {test_ds.column_names}")

train examples: 1798, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']
test examples: 316, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']


In [5]:
# remove cols
cols = ["dataset", "class_label", "test_fastaid", "L_seq", 'source_description', 'Genome type', 'Family', 'subds_seqid', "__index_level_0__"]

train_ds = train_ds.remove_columns(cols)
test_ds = test_ds.remove_columns(cols)

In [6]:
# rename y to label
train_ds = train_ds.rename_column("y", "label")
train_ds = train_ds.rename_column("source_fasta_id", "fasta_id")
test_ds = test_ds.rename_column("y", "label")
test_ds = test_ds.rename_column("source_fasta_id", "fasta_id")

In [7]:
train_df = train_ds.to_pandas()
test_df = test_ds.to_pandas()

In [8]:
train_df.head()

,sequence_id,label,fasta_id,orientation,sequence
0,0,0,PHAA0000323,forward,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...
1,1,0,PHAA0000323,reverse,TAACCGACCGGTCAGGAAGGTACCCCTGGGGGGGTACTCCCCCTGG...
2,2,1,PHAA0000761,forward,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...
3,3,1,PHAA0000761,reverse,GGGCGCAGGATACCAGCAGGGTTGACTTACCCGACTCACTCGGGTA...
4,4,0,PHAA0000840,forward,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...


In [9]:
test_df.head()

,sequence_id,label,fasta_id,orientation,sequence
0,1798,1,PHAA0000415,forward,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...
1,1799,1,PHAA0000415,reverse,ATGTACAATATTAAATGCCTGACCAAAAACGAACAAGCTGAAATTG...
2,1800,0,PHAA0000088,forward,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...
3,1801,0,PHAA0000088,reverse,CATGAAGCATTAAAAACACAGAATTACCGAAGATCGTCCAAGAGCA...
4,1802,0,PHAA0000253,forward,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...


In [10]:
def save_pandas_as_fasta(df: pd.DataFrame, file_path: str) -> None:
    records = []
    for _, row in df.iterrows():
        records.append(SeqRecord(
            Seq(row["sequence"]),
            id=str(row["fasta_id"]),
            description=f"empy description",
        ))
    SeqIO.write(records, file_path, "fasta")

In [11]:
train_save_path = f"{base_dir}/example-train-dataset/phastyle-sequencedb-bachlip-training.fasta"
test_save_path = f"{base_dir}/example-test-dataset/phastyle-sequencedb-bachlip-validation.fasta"

In [12]:
save_pandas_as_fasta(train_df, train_save_path)
save_pandas_as_fasta(test_df, test_save_path)

In [13]:
train_dataset = load_contigs(
    [train_save_path],
    IsAddHeader=True,
    adding_reverse_complement=False,
    AsDataFrame=True,
    to_uppercase=True,
    is_add_sequence_id=True,
)

2026-10-08 11:43:40,947 - INFO - Loading sequence data into memory!


In [14]:
train_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation
0,0,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,forward
1,1,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TAACCGACCGGTCAGGAAGGTACCCCTGGGGGGGTACTCCCCCTGG...,forward
2,2,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,forward
3,3,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,GGGCGCAGGATACCAGCAGGGTTGACTTACCCGACTCACTCGGGTA...,forward
4,4,PHAA0000840,PHAA0000840 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,forward


In [15]:
train_dataset[train_dataset["fasta_id"] == "PHAA0000323"]

,sequence_id,fasta_id,description,source_file,sequence,orientation
0,0,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,forward
1,1,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TAACCGACCGGTCAGGAAGGTACCCCTGGGGGGGTACTCCCCCTGG...,forward


In [16]:
labels = train_df[["fasta_id", "label"]].drop_duplicates()
train_dataset = train_dataset.merge(labels, on="fasta_id", how="left", validate="many_to_one")

In [17]:
train_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation,label
0,0,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,forward,0
1,1,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TAACCGACCGGTCAGGAAGGTACCCCTGGGGGGGTACTCCCCCTGG...,forward,0
2,2,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,forward,1
3,3,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,GGGCGCAGGATACCAGCAGGGTTGACTTACCCGACTCACTCGGGTA...,forward,1
4,4,PHAA0000840,PHAA0000840 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,forward,0


In [19]:
test_dataset = load_contigs(
    [test_save_path],
    IsAddHeader=True,
    adding_reverse_complement=False,
    AsDataFrame=True,
    to_uppercase=True,
    is_add_sequence_id=True,
)

2026-10-08 11:43:56,396 - INFO - Loading sequence data into memory!


In [20]:
test_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation
0,0,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...,forward
1,1,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGTACAATATTAAATGCCTGACCAAAAACGAACAAGCTGAAATTG...,forward
2,2,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward
3,3,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,CATGAAGCATTAAAAACACAGAATTACCGAAGATCGTCCAAGAGCA...,forward
4,4,PHAA0000253,PHAA0000253 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward


In [21]:
labels = test_df[["fasta_id", "label"]].drop_duplicates()
test_dataset = test_dataset.merge(labels, on="fasta_id", how="left", validate="many_to_one")

In [22]:
test_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation,label
0,0,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...,forward,1
1,1,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGTACAATATTAAATGCCTGACCAAAAACGAACAAGCTGAAATTG...,forward,1
2,2,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward,0
3,3,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,CATGAAGCATTAAAAACACAGAATTACCGAAGATCGTCCAAGAGCA...,forward,0
4,4,PHAA0000253,PHAA0000253 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward,0


In [32]:
# import importlib
# import prokbert.sequtils
# importlib.reload(prokbert.sequtils)
# from prokbert.sequtils import segment_sequences

segmentation_params = {
    "max_length": 2_000,
    "min_length": int(2_000 * 0.5),
    "type": "random",
    "coverage": 1
}
raw_segment_train_df = segment_sequences(train_dataset, segmentation_params, AsDataFrame=True)

2026-10-08 11:46:49,650 - INFO - Checking input DataFrame!
2026-10-08 11:46:49,652 - INFO - Checking input sequence_id is valid primary key in the DataFrame
2026-10-08 11:46:49,740 - INFO - Sampling 58341 segments from 1798 sequences.
2026-10-08 11:46:49,814 - INFO - Too short segment, skip! Sampled segment: CTCAGCGGTCGCCAGGCGGGGCAGCTCGATGCGCGTCAGCGTGGTCACTCGGTGAACCCGTCAGGGTCGGCGGCGTCCAGGAACTCGTCGGCCTGGGCGTCCAGCTCGGCGGGCGTCGGCTCGATGCCCGGCGCGGTCTCACCGCTCCACTCGGCCAGCAGCTCCAGCAGCGCCTCGCTGTTGGTCTTGAGACCGCGCTCCGTGCGAAAGGCCTCCAGCTTCTCCTGAGACCAGACGAACGCCGGGATCGGCATGGTCAGCACGATCATCCGGTTGTTCGCGTTCTCGGCGTACTCGTCGCGCCGCTGGGTCATGTCCTTGGCGTCGATCAGGCCGTCCTCGCGCTTGCCCTTGGGGCTCGGCTCGGGCTCGTCCTGAGGGCTGACCGGTGGCAGATCCTGTTCCTCCAGCAGCGCGGTCAGATCCTCCACGTCCGCCTCGGTGAAGCCCAGGCCCTCGATGTCCGCGCCGAACCCGGTGACCAGCTCGGCCAGCTCGATCTCATCGAACCCGCCCAGCTGGCCGGTCTGGTTGTCCGCGACCACGATGCGCTCGGCCATGTCGTCATCGACGTCCACCCAGTGGACCAGGATCTTCTGCCAGGCGGGGTCGTCGGGCTCGGCCTCGGCCAGCTCGCGGAACGCCATCAGGGTGTGGTTACCGGCCAGGATCTCCAGCGGTCGGCCCGTG

In [33]:
raw_segment_train_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,415,2415,CTTATATTATGAGGGCCCAAGAGGGCCCTATCTAGAGCGCCACCAG...
1,1,0,2132,4132,CAAGAACGACGAAGGTGAATGGGAGATCGAAGTGGAGGTGCTGAAG...
2,2,0,3235,5235,TGACCGACATGCTCAAGCAGCGCCTCGCCCAAGCTCAGGCGAGCGG...
3,3,0,3924,5924,TCGTGACCTTCGCTGTCGACGTGACCAGCACGGCCGTCAAGAACTT...
4,4,0,6965,8965,GCAGGCTCAAGAAGGTCGTCATGTGGGGAAATCCCATGCGGCAGAA...


In [34]:
labels = train_dataset.set_index("sequence_id")["label"]
assert labels.index.is_unique, "sequence_id is not unique in test_dataset"

raw_segment_train_df["label"] = raw_segment_train_df["sequence_id"].map(labels)
assert raw_segment_train_df["label"].notna().all(), "some segments have a sequence_id missing from test_dataset"

In [35]:
raw_segment_train_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment,label
0,0,0,415,2415,CTTATATTATGAGGGCCCAAGAGGGCCCTATCTAGAGCGCCACCAG...,0
1,1,0,2132,4132,CAAGAACGACGAAGGTGAATGGGAGATCGAAGTGGAGGTGCTGAAG...,0
2,2,0,3235,5235,TGACCGACATGCTCAAGCAGCGCCTCGCCCAAGCTCAGGCGAGCGG...,0
3,3,0,3924,5924,TCGTGACCTTCGCTGTCGACGTGACCAGCACGGCCGTCAAGAACTT...,0
4,4,0,6965,8965,GCAGGCTCAAGAAGGTCGTCATGTGGGGAAATCCCATGCGGCAGAA...,0


In [36]:
segmentation_params = {
    "max_length": 2_000,
    "min_length": int(2_000 * 0.5),
    "type": "contiguous",
}

raw_segment_test_df = segment_sequences(test_dataset, segmentation_params, AsDataFrame=True)

2026-10-08 11:47:06,379 - INFO - Checking input DataFrame!
2026-10-08 11:47:06,381 - INFO - Checking input sequence_id is valid primary key in the DataFrame


In [37]:
raw_segment_test_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,0,2000,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...
1,1,0,2000,4000,TTCAGGATTTCCATTAAGTGCATGAGCATCAATCATGTTAGTAATC...
2,2,0,4000,6000,TTGAGAACTTTACCACGCAATGGATAACCACCGTGAAGTTCTTTAT...
3,3,0,6000,8000,CACTTCATCAAAGAACCCAGCTTCGTTAGTAGAGAGCCAGATAAAG...
4,4,0,8000,10000,AAATTAGGTCTGTCATAACCTACATGTAATAACCACTCTTCACTCC...


In [38]:
labels = test_dataset.set_index("sequence_id")["label"]
assert labels.index.is_unique, "sequence_id is not unique in test_dataset"

raw_segment_test_df["label"] = raw_segment_test_df["sequence_id"].map(labels)
assert raw_segment_test_df["label"].notna().all(), "some segments have a sequence_id missing from test_dataset"

In [39]:
raw_segment_test_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment,label
0,0,0,0,2000,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...,1
1,1,0,2000,4000,TTCAGGATTTCCATTAAGTGCATGAGCATCAATCATGTTAGTAATC...,1
2,2,0,4000,6000,TTGAGAACTTTACCACGCAATGGATAACCACCGTGAAGTTCTTTAT...,1
3,3,0,6000,8000,CACTTCATCAAAGAACCCAGCTTCGTTAGTAGAGAGCCAGATAAAG...,1
4,4,0,8000,10000,AAATTAGGTCTGTCATAACCTACATGTAATAACCACTCTTCACTCC...,1


In [41]:
hf_train_dataset = datasets.Dataset.from_pandas(raw_segment_train_df)
hf_test_dataset = datasets.Dataset.from_pandas(raw_segment_test_df)

In [ ]:
model_path = 'neuralbioinfo/prokbert-mini-long'

model = ProkBertForSequenceClassification.from_pretrained(model_path, trust_remote_code=True)
tokenizer = LCATokenizer.from_pretrained(model_path, trust_remote_code=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

2026-10-08 11:47:35,832 - INFO - HTTP Request: HEAD https://huggingface.co/neuralbioinfo/PhaStyle-mini/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-08 11:47:35,868 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralbioinfo/PhaStyle-mini/85ec53cdc7676fef2ea5b8b2a543424b8ca45791/config.json "HTTP/1.1 200 OK"
2026-10-08 11:47:36,367 - INFO - HTTP Request: HEAD https://huggingface.co/neuralbioinfo/PhaStyle-mini/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-10-08 11:47:36,711 - INFO - HTTP Request: HEAD https://huggingface.co/neuralbioinfo/PhaStyle-mini/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-08 11:47:36,745 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralbioinfo/PhaStyle-mini/85ec53cdc7676fef2ea5b8b2a543424b8ca45791/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-08 11:47:36,893 - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralbioinfo/PhaS

In [43]:
max_length=512

def _tokenize_fn(batch):
    tokenized = tokenizer(
        batch["segment"],
        padding="longest",
        truncation=True,
        max_length=max_length,
    )
    # Zero out first/last attention token
    masks = tokenized["attention_mask"]
    for m in masks:
        m[0] = 0
        m[-1] = 0
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": masks,
        "labels": batch["label"]
    }

In [44]:
tokenized_train_ds = hf_train_dataset.map(
    _tokenize_fn,
    batched=True,
    num_proc=num_cores,
    remove_columns=hf_train_dataset.column_names,
    keep_in_memory=True,
)
tokenized_test_ds = hf_test_dataset.map(
    _tokenize_fn,
    batched=True,
    num_proc=num_cores,
    remove_columns=hf_test_dataset.column_names,
    keep_in_memory=True,
)

Map (num_proc=8):   0%|          | 0/57380 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/12420 [00:00<?, ? examples/s]

In [45]:
tokenized_train_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 57380
})

In [49]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):   # some models also return hidden states
        logits = logits[0]
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

In [50]:
training_args = TrainingArguments(
    output_dir=f"{base_dir}/phastyle-sequence-db-embedding-v0-compare",
    max_steps=20_000,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 4,
    learning_rate=5e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    dataloader_num_workers=4,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# srun --job-name=phastyle-sequence-db-embedding-v0-compare \
#      --nodes=1 \
#      --nodelist=lolka \
#      --ntasks-per-node=1 \
#      --cpus-per-task=8 \
#      --gres=gpu:2 \
#      --mem=150G \
#      --time=7-00:00:00 \
#      --partition=hi \
#      --output=/home/osvathm/prokbert/logs/%x_%j.out \
#      --error=/home/osvathm/prokbert/logs/%x_%j.err \
#      torchrun --nproc_per_node=2 examples/phastyle_training_compare.py

# notes
# requested gpu count should match nproc_per_node param